# counseLLM — Training Notebook

Interactive notebook for running the SFT + DPO training pipeline.
Useful for running on Colab/Kaggle or step-by-step debugging.

**Pipeline:**
1. Data preparation (SFT + DPO)
2. Stage 1: SFT with QLoRA
3. Stage 2: DPO alignment
4. Merge adapters

In [ ]:
!pip install torch transformers trl peft bitsandbytes datasets accelerate wandb pyyaml -q

In [ ]:
import torch
import yaml
from pathlib import Path
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer, DPOConfig, DPOTrainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Login to HuggingFace (needed for Llama 3.1 gated model)
from huggingface_hub import login
login()  # paste your HF token when prompted

## Step 0: Prepare Data

Run the data preparation scripts if you haven't already.

In [ ]:
!python ../data/prepare_sft_data.py --skip-psych8k
!python ../data/prepare_dpo_data.py --sample-size 2000 --min-rating-gap 3

## Step 1: Load Config

In [ ]:
PROJECT_ROOT = Path("..").resolve()

with open(PROJECT_ROOT / "configs" / "sft_config.yaml") as f:
    sft_cfg = yaml.safe_load(f)

with open(PROJECT_ROOT / "configs" / "dpo_config.yaml") as f:
    dpo_cfg = yaml.safe_load(f)

BASE_MODEL = sft_cfg["model"]["name"]
print(f"Base model: {BASE_MODEL}")

## Step 2: SFT Training

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load SFT data
sft_dataset = load_dataset(
    "json",
    data_files={
        "train": str(PROJECT_ROOT / sft_cfg["data"]["train_file"]),
        "validation": str(PROJECT_ROOT / sft_cfg["data"]["val_file"]),
    },
)
print(f"SFT Train: {len(sft_dataset['train'])} | Val: {len(sft_dataset['validation'])}")

In [ ]:
# Load model in 4-bit
q_cfg = sft_cfg["quantization"]
bnb_config = BitsAndBytesConfig(
    load_in_4bit=q_cfg["load_in_4bit"],
    bnb_4bit_quant_type=q_cfg["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=getattr(torch, q_cfg["bnb_4bit_compute_dtype"]),
    bnb_4bit_use_double_quant=q_cfg["bnb_4bit_use_double_quant"],
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=sft_cfg["model"]["attn_implementation"],
    torch_dtype=getattr(torch, sft_cfg["model"]["torch_dtype"]),
)
model = prepare_model_for_kbit_training(model)

# Apply LoRA
lora_cfg = sft_cfg["lora"]
lora_config = LoraConfig(
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    target_modules=lora_cfg["target_modules"],
    bias=lora_cfg["bias"],
    task_type=lora_cfg["task_type"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Configure and run SFT training
t_cfg = sft_cfg["training"]
l_cfg = sft_cfg["logging"]
sft_output_dir = PROJECT_ROOT / sft_cfg["data"]["output_dir"]

training_args = SFTConfig(
    output_dir=str(sft_output_dir),
    num_train_epochs=t_cfg["num_train_epochs"],
    per_device_train_batch_size=t_cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=t_cfg["per_device_eval_batch_size"],
    gradient_accumulation_steps=t_cfg["gradient_accumulation_steps"],
    learning_rate=t_cfg["learning_rate"],
    lr_scheduler_type=t_cfg["lr_scheduler_type"],
    warmup_ratio=t_cfg["warmup_ratio"],
    max_seq_length=t_cfg["max_seq_length"],
    bf16=t_cfg["bf16"],
    gradient_checkpointing=t_cfg["gradient_checkpointing"],
    seed=t_cfg["seed"],
    logging_steps=l_cfg["logging_steps"],
    eval_strategy=l_cfg["eval_strategy"],
    eval_steps=l_cfg["eval_steps"],
    save_strategy=l_cfg["save_strategy"],
    save_steps=l_cfg["save_steps"],
    save_total_limit=l_cfg["save_total_limit"],
    load_best_model_at_end=l_cfg["load_best_model_at_end"],
    metric_for_best_model=l_cfg["metric_for_best_model"],
    greater_is_better=l_cfg["greater_is_better"],
    report_to=l_cfg["report_to"],
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["validation"],
    processing_class=tokenizer,
)

print("Starting SFT training...")
trainer.train()

In [ ]:
# Save SFT adapter
sft_final_dir = sft_output_dir / "final"
trainer.save_model(str(sft_final_dir))
tokenizer.save_pretrained(str(sft_final_dir))
print(f"SFT adapter saved to: {sft_final_dir}")

# Free memory before DPO
del model, trainer
torch.cuda.empty_cache()

## Step 3: Quick SFT Sanity Check

Generate a response before moving to DPO.

In [ ]:
# Quick inference check on SFT model
sft_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
sft_model = PeftModel.from_pretrained(sft_model, str(sft_final_dir))
sft_model.eval()

test_prompt = "I've been feeling really anxious lately and I can't sleep at night. My mind keeps racing with worries about work and relationships."

messages = [
    {"role": "system", "content": sft_cfg["system_prompt"]},
    {"role": "user", "content": test_prompt},
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(sft_model.device)

with torch.no_grad():
    outputs = sft_model.generate(**inputs, max_new_tokens=300, temperature=0.7, top_p=0.9, do_sample=True)

generated = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(generated, skip_special_tokens=True)

print("USER:", test_prompt)
print("\nSFT MODEL:", response)

del sft_model
torch.cuda.empty_cache()

## Step 4: DPO Training

In [ ]:
# Load DPO data
dpo_dataset = load_dataset(
    "json",
    data_files={
        "train": str(PROJECT_ROOT / dpo_cfg["data"]["train_file"]),
        "validation": str(PROJECT_ROOT / dpo_cfg["data"]["val_file"]),
    },
)
print(f"DPO Train: {len(dpo_dataset['train'])} | Val: {len(dpo_dataset['validation'])}")

# Load base model + merge SFT adapter
tokenizer.padding_side = "left"  # DPO requires left padding

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=dpo_cfg["model"]["attn_implementation"],
    torch_dtype=getattr(torch, dpo_cfg["model"]["torch_dtype"]),
)

print("Merging SFT adapter...")
model = PeftModel.from_pretrained(model, str(sft_final_dir))
model = model.merge_and_unload()
model = prepare_model_for_kbit_training(model)

# New LoRA for DPO
dpo_lora_cfg = dpo_cfg["lora"]
dpo_lora_config = LoraConfig(
    r=dpo_lora_cfg["r"],
    lora_alpha=dpo_lora_cfg["lora_alpha"],
    lora_dropout=dpo_lora_cfg["lora_dropout"],
    target_modules=dpo_lora_cfg["target_modules"],
    bias=dpo_lora_cfg["bias"],
    task_type=dpo_lora_cfg["task_type"],
)
model.add_adapter(dpo_lora_config)
model.print_trainable_parameters()

In [ ]:
# Run DPO training
dt_cfg = dpo_cfg["training"]
dl_cfg = dpo_cfg["logging"]
dpo_output_dir = PROJECT_ROOT / dpo_cfg["data"]["output_dir"]

dpo_training_args = DPOConfig(
    output_dir=str(dpo_output_dir),
    beta=dpo_cfg["dpo"]["beta"],
    num_train_epochs=dt_cfg["num_train_epochs"],
    per_device_train_batch_size=dt_cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=dt_cfg["per_device_eval_batch_size"],
    gradient_accumulation_steps=dt_cfg["gradient_accumulation_steps"],
    learning_rate=dt_cfg["learning_rate"],
    lr_scheduler_type=dt_cfg["lr_scheduler_type"],
    warmup_ratio=dt_cfg["warmup_ratio"],
    max_length=dt_cfg["max_length"],
    max_prompt_length=dt_cfg["max_prompt_length"],
    bf16=dt_cfg["bf16"],
    gradient_checkpointing=dt_cfg["gradient_checkpointing"],
    seed=dt_cfg["seed"],
    logging_steps=dl_cfg["logging_steps"],
    eval_strategy=dl_cfg["eval_strategy"],
    eval_steps=dl_cfg["eval_steps"],
    save_strategy=dl_cfg["save_strategy"],
    save_steps=dl_cfg["save_steps"],
    save_total_limit=dl_cfg["save_total_limit"],
    load_best_model_at_end=dl_cfg["load_best_model_at_end"],
    metric_for_best_model=dl_cfg["metric_for_best_model"],
    greater_is_better=dl_cfg["greater_is_better"],
    report_to=dl_cfg["report_to"],
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_training_args,
    train_dataset=dpo_dataset["train"],
    eval_dataset=dpo_dataset["validation"],
    processing_class=tokenizer,
)

print("Starting DPO training...")
dpo_trainer.train()

In [ ]:
# Save DPO adapter
dpo_final_dir = dpo_output_dir / "final"
dpo_trainer.save_model(str(dpo_final_dir))
tokenizer.save_pretrained(str(dpo_final_dir))
print(f"DPO adapter saved to: {dpo_final_dir}")

del model, dpo_trainer
torch.cuda.empty_cache()

## Step 5: Merge Adapters & Push to Hub

In [ ]:
# Merge both adapters into base model (full precision)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Merge SFT
model = PeftModel.from_pretrained(model, str(sft_final_dir))
model = model.merge_and_unload()

# Merge DPO
model = PeftModel.from_pretrained(model, str(dpo_final_dir))
model = model.merge_and_unload()

# Save merged model
merged_dir = PROJECT_ROOT / "outputs" / "merged" / "sft-dpo"
model.save_pretrained(str(merged_dir))
tokenizer.save_pretrained(str(merged_dir))
print(f"Merged model saved to: {merged_dir}")

In [ ]:
# Optional: Push to HuggingFace Hub
HUB_REPO = "your-username/counseLLM"  # <-- change this

# model.push_to_hub(HUB_REPO)
# tokenizer.push_to_hub(HUB_REPO)
# print(f"Pushed to: https://huggingface.co/{HUB_REPO}")

print(f"Uncomment the lines above and set HUB_REPO to push to HuggingFace Hub")